# 01 · Capa Bronze — Extracción y carga cruda

**IDY1102 · EP3 · DataRetail 360 · Proceso Ventas**

| Campo | Detalle |
|---|---|
| **Capa Medallion** | Bronze (datos crudos, tal como llegan) |
| **Etapa ETL** | EXTRACT (sección 1 de la plantilla 2.2.3) |
| **IL** | IL 2.3 (pipeline) · IL 2.4 (almacenamiento trazable) |
| **Entrada** | Archivos CSV de `data/raw` (uno por fuente y lote) |
| **Salida** | Tablas `bronze_*` en `db/dataretail360.db` + bitácora `etl_control` |

**Reglas de esta capa**
1. Los datos se guardan **sin modificar**: todo como texto, incluso los errores. Así siempre se puede volver al dato original.
2. Cada fila recibe columnas de **trazabilidad**: lote, archivo de origen, número de fila y fecha de carga.
3. **Recarga controlada (idempotencia):** si un lote se vuelve a cargar, primero se borran sus filas anteriores.

## 0. Configuración

In [14]:
import sys
from pathlib import Path

import pandas as pd
from sqlalchemy import text

# Permite importar los módulos de la carpeta src/
RAIZ = Path.cwd().parent
sys.path.append(str(RAIZ / "src"))

import config
import utils

# >>> Lote a procesar (1 = enero-junio, 2 = julio-septiembre) <<<
ID_LOTE = 2

logger = utils.configurar_logging("bronze")
motor = utils.obtener_motor()
fuentes = config.FUENTES_POR_LOTE[ID_LOTE]

logger.info(f"=== Inicio capa Bronze | lote {ID_LOTE} | {len(fuentes)} fuentes ===")

2026-10-25 14:38:14 | INFO    | bronze | === Inicio capa Bronze | lote 2 | 4 fuentes ===


## 1. EXTRACT — leer las fuentes

Se leen los CSV **como texto** (`dtype=str`) y sin convertir vacíos en nulos (`keep_default_na=False`). Así Bronze conserva exactamente lo que entregó cada sistema: si el POS envió `2026-02-30`, eso queda guardado, y la capa Silver decidirá qué hacer.

**Manejo de errores:**
- Si falta un archivo → **error fatal**: se detiene el pipeline, porque cargar un lote incompleto dejaría datos inconsistentes.
- Si un archivo viene vacío → **advertencia**: se registra y se continúa con las demás fuentes.

In [15]:
def extraer_csv(nombre_fuente: str, archivo: str, id_lote: int) -> pd.DataFrame:
    """Lee un CSV de data/raw sin modificarlo y agrega columnas de trazabilidad."""
    ruta = config.CARPETA_RAW / archivo
    df = pd.read_csv(ruta, dtype=str, keep_default_na=False, encoding="utf-8")

    # Columnas de trazabilidad (prefijo _ para distinguirlas de los datos del negocio)
    df["_id_lote"] = id_lote
    df["_archivo_origen"] = archivo
    df["_fila_origen"] = df.index + 2          # +2: la fila 1 del CSV es el encabezado
    df["_fecha_carga"] = pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S")
    return df


extraidos = {}
for nombre_fuente, archivo in fuentes.items():
    try:
        df = extraer_csv(nombre_fuente, archivo, ID_LOTE)
        extraidos[nombre_fuente] = df
        logger.info(f"Extraído {archivo}: {len(df)} filas")
    except FileNotFoundError:
        logger.error(f"No se encontró {archivo} en data/raw. Se detiene el pipeline.")
        raise
    except pd.errors.EmptyDataError:
        logger.warning(f"{archivo} está vacío. Se omite esta fuente.")

logger.info(f"Extracción terminada: {len(extraidos)} de {len(fuentes)} fuentes")

2026-10-25 14:38:14 | INFO    | bronze | Extraído productos_lote2.csv: 35 filas
2026-10-25 14:38:14 | INFO    | bronze | Extraído clientes_lote2.csv: 430 filas
2026-10-25 14:38:14 | INFO    | bronze | Extraído ventas_tienda_lote2.csv: 3765 filas
2026-10-25 14:38:14 | INFO    | bronze | Extraído ventas_online_lote2.csv: 1198 filas
2026-10-25 14:38:14 | INFO    | bronze | Extracción terminada: 4 de 4 fuentes


## 2. LOAD — cargar a Bronze con recarga controlada

Para cada fuente se crea (o actualiza) la tabla `bronze_<fuente>`. Todo ocurre dentro de una **transacción** (`motor.begin()`): si algo falla a mitad de camino, la base vuelve al estado anterior y no quedan cargas a medias.

Cada carga queda anotada en la tabla **`etl_control`**, la bitácora del pipeline.

In [16]:
def cargar_bronze(df: pd.DataFrame, nombre_fuente: str, id_lote: int) -> int:
    """Carga un DataFrame en bronze_<fuente>, reemplazando solo las filas del mismo lote."""
    tabla = f"bronze_{nombre_fuente}"
    with motor.begin() as conexion:
        if utils.tabla_existe(motor, tabla):
            resultado = conexion.execute(
                text(f"DELETE FROM {tabla} WHERE _id_lote = :lote"), {"lote": id_lote}
            )
            if resultado.rowcount > 0:
                logger.warning(f"{tabla}: el lote {id_lote} ya existía; "
                               f"se borraron {resultado.rowcount} filas antes de recargar")
        df.to_sql(tabla, conexion, if_exists="append", index=False)
        utils.registrar_control(conexion, id_lote, "bronze", tabla,
                                df["_archivo_origen"].iloc[0], len(df), "OK")
    logger.info(f"Cargado {tabla}: {len(df)} filas")
    return len(df)


for nombre_fuente, df in extraidos.items():
    try:
        cargar_bronze(df, nombre_fuente, ID_LOTE)
    except Exception as error:
        logger.error(f"Falló la carga de {nombre_fuente}: {error}")
        raise

2026-10-25 14:38:14 | INFO    | bronze | Cargado bronze_productos: 35 filas
2026-10-25 14:38:14 | INFO    | bronze | Cargado bronze_clientes: 430 filas
2026-10-25 14:38:15 | INFO    | bronze | Cargado bronze_ventas_tienda: 3765 filas
2026-10-25 14:38:15 | INFO    | bronze | Cargado bronze_ventas_online: 1198 filas


## 3. VALIDACIÓN — ¿llegó todo?

Se compara la cantidad de filas de cada CSV con la cantidad de filas cargadas en Bronze para este lote. Si alguna no coincide, el `assert` detiene el notebook con un mensaje claro.

In [17]:
resumen = []
with motor.connect() as conexion:
    for nombre_fuente, df in extraidos.items():
        tabla = f"bronze_{nombre_fuente}"
        en_bd = conexion.execute(
            text(f"SELECT COUNT(*) FROM {tabla} WHERE _id_lote = :lote"), {"lote": ID_LOTE}
        ).scalar()
        resumen.append({"tabla": tabla, "filas_csv": len(df), "filas_bronze": en_bd,
                        "coincide": len(df) == en_bd})

resumen = pd.DataFrame(resumen)
assert resumen["coincide"].all(), "Hay tablas Bronze con filas faltantes o sobrantes"
logger.info(f"Validación OK: {resumen['filas_bronze'].sum()} filas cargadas en Bronze")
resumen

2026-10-25 14:38:15 | INFO    | bronze | Validación OK: 5428 filas cargadas en Bronze


,tabla,filas_csv,filas_bronze,coincide
0,bronze_productos,35,35,True
1,bronze_clientes,430,430,True
2,bronze_ventas_tienda,3765,3765,True
3,bronze_ventas_online,1198,1198,True


## 4. Prueba de idempotencia

Se ejecuta la carga **una segunda vez** y se verifica que los conteos no cambian. Si el pipeline no fuera idempotente, las filas se duplicarían.

In [18]:
for nombre_fuente, df in extraidos.items():
    cargar_bronze(df, nombre_fuente, ID_LOTE)

with motor.connect() as conexion:
    for _, fila in resumen.iterrows():
        en_bd = conexion.execute(
            text(f"SELECT COUNT(*) FROM {fila['tabla']} WHERE _id_lote = :lote"), {"lote": ID_LOTE}
        ).scalar()
        assert en_bd == fila["filas_bronze"], f"{fila['tabla']} se duplicó al recargar"

logger.info("Idempotencia OK: recargar el lote no duplicó filas")

2026-10-25 14:38:15 | WARNING | bronze | bronze_productos: el lote 2 ya existía; se borraron 35 filas antes de recargar
2026-10-25 14:38:15 | INFO    | bronze | Cargado bronze_productos: 35 filas
2026-10-25 14:38:15 | WARNING | bronze | bronze_clientes: el lote 2 ya existía; se borraron 430 filas antes de recargar
2026-10-25 14:38:15 | INFO    | bronze | Cargado bronze_clientes: 430 filas
2026-10-25 14:38:15 | WARNING | bronze | bronze_ventas_tienda: el lote 2 ya existía; se borraron 3765 filas antes de recargar
2026-10-25 14:38:15 | INFO    | bronze | Cargado bronze_ventas_tienda: 3765 filas
2026-10-25 14:38:15 | WARNING | bronze | bronze_ventas_online: el lote 2 ya existía; se borraron 1198 filas antes de recargar
2026-10-25 14:38:15 | INFO    | bronze | Cargado bronze_ventas_online: 1198 filas
2026-10-25 14:38:15 | INFO    | bronze | Idempotencia OK: recargar el lote no duplicó filas


## 5. Bitácora de cargas (`etl_control`)

Evidencia de trazabilidad: cada ejecución queda registrada con su lote, tabla, archivo de origen, filas y fecha.

In [19]:
bitacora = pd.read_sql("SELECT * FROM etl_control ORDER BY fecha_ejecucion DESC", motor)
logger.info(f"=== Fin capa Bronze | lote {ID_LOTE} ===")
bitacora.head(15)

2026-10-25 14:38:15 | INFO    | bronze | === Fin capa Bronze | lote 2 ===


,id_lote,capa,tabla,origen,filas,estado,fecha_ejecucion
0,2,bronze,bronze_ventas_tienda,ventas_tienda_lote2.csv,3765,OK,2026-10-25 14:38:15
1,2,bronze,bronze_ventas_online,ventas_online_lote2.csv,1198,OK,2026-10-25 14:38:15
2,2,bronze,bronze_productos,productos_lote2.csv,35,OK,2026-10-25 14:38:15
3,2,bronze,bronze_clientes,clientes_lote2.csv,430,OK,2026-10-25 14:38:15
4,2,bronze,bronze_ventas_tienda,ventas_tienda_lote2.csv,3765,OK,2026-10-25 14:38:15
5,2,bronze,bronze_ventas_online,ventas_online_lote2.csv,1198,OK,2026-10-25 14:38:15
6,2,bronze,bronze_productos,productos_lote2.csv,35,OK,2026-10-25 14:38:14
7,2,bronze,bronze_clientes,clientes_lote2.csv,430,OK,2026-10-25 14:38:14
8,1,bronze,bronze_ventas_tienda,ventas_tienda_lote1.csv,7452,OK,2026-10-25 14:33:35
9,1,bronze,bronze_ventas_online,ventas_online_lote1.csv,2367,OK,2026-10-25 14:33:35
